In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
%cd /workspace/Code/experimental/

/workspace/Code/experimental


In [6]:
from data.datamodule import TextDataModule
from src.modules.core.encoder import TransformerEncoder
from src.modules.mlm_baseline.mlm_module import MLMModule
import lightning as L
from lightning.pytorch.loggers import WandbLogger
from data.tokenizer import TokenizerWrapper

In [7]:
dm = TextDataModule(
    train_path="data/raw/train_10000.parquet",
    val_path="data/raw/val_1000.parquet",
    tokenizer_name="bert-base-uncased",
    window_size=128,
    alpha=0.5,
    batch_size=32,
)

encoder = TransformerEncoder(
    vocab_size=30522,   # bert-base-uncased
    d_model=8,
    n_heads=1,
    n_layers=4,
    d_ff=256,
    max_seq_len=128,
)

module = MLMModule(encoder=encoder, vocab_size=30522, d_model=8, lr=3e-5)

In [8]:
dm.setup()

Chargement du tokenizer depuis le cache local : ./data/tokenizers/bert-base-uncased


In [9]:
batch_iter = iter(dm.train_dataloader())
first_batch = next(batch_iter)

/workspace/Code/experimental/data/masking.py:89: UserWarning: alpha=0.5 would mask 1/1 tokens — all tokens will be masked
  warnings.warn(


In [10]:
print(first_batch.keys())
print(first_batch['input_ids'])
print(first_batch['attention_mask'])
print(first_batch['masked_positions'])
print(first_batch['target_ids'])

dict_keys(['input_ids', 'attention_mask', 'masked_positions', 'target_ids'])
tensor([[ 101,  103,  103,  ..., 2800, 2182,  103],
        [   0,    0,    0,  ...,    0,    0,  102],
        [ 101,  103,  103,  ...,  103,  103, 1010],
        ...,
        [ 101, 2007, 2936,  ...,  103,  103, 2115],
        [3269,  103,  103,  ..., 8962,  103,  103],
        [ 103, 1024,  103,  ..., 2000, 7633, 1996]])
tensor([[1, 1, 1,  ..., 1, 1, 1],
        [0, 0, 0,  ..., 0, 0, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])
tensor([[ 1,  2,  3,  ..., -1, -1, -1],
        [-1, -1, -1,  ..., -1, -1, -1],
        [ 1,  2,  3,  ..., -1, -1, -1],
        ...,
        [ 3,  6,  8,  ..., -1, -1, -1],
        [ 1,  2,  3,  ..., -1, -1, -1],
        [ 0,  5,  4,  ..., -1, -1, -1]])
tensor([[25088,  1996,  2088,  ...,     0,     0,     0],
        [    0,     0,     0,  ...,     0,     0,     0],
        [

In [11]:
tokenizer = TokenizerWrapper()

Chargement du tokenizer depuis le cache local : ./data/tokenizers/bert-base-uncased


In [12]:
print(tokenizer.decode(first_batch['input_ids'][16], False))
print(tokenizer.decode(first_batch['target_ids'][16], False))

[MASK] enables [MASK] [MASK] [MASK] [MASK] [MASK] tiny [MASK] [MASK] rocks [MASK] [MASK] survive [MASK] [MASK] [MASK] long [MASK] of [MASK] [MASK] another special [MASK] [MASK] ridge [MASK] s mountain is a series of upland [MASK] [MASK] [MASK] [MASK] [MASK] these areas hold [MASK] [MASK] the winter and spring [MASK] but [MASK] [MASK] during [MASK] [MASK] [MASK] — [MASK] they have [MASK] [MASK]. these pools [MASK] excellent breeding habitat [MASK] many amphibians [MASK] [MASK] several frog and [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] 2000 [MASK] the [MASK] zoo [MASK] the [MASK] [MASK] based [MASK] land [MASK] [MASK] [MASK] [MASK] [MASK] together to [MASK] 180 [MASK] along the [SEP]
this the fern to grow in cracks in and to without water for periods time. feature of ’ depression pools and swamps. water during, dry up the summer months thus no fish provide for, including

In [13]:
i = 0
prc_arr = []
for batch in dm.train_dataloader():
    i+=1
    text = dm.tokenizer.decode(batch["input_ids"], skip_special_tokens=False)[0]
    hide_prc : float = (text.count('[MASK]')) / len(batch["input_ids"][0]) 
    print(f"--- Chunk {i} --- MASK represents {hide_prc}")
    print(text)
    print()
    prc_arr.append(hide_prc)
    if i > 20:
        break


print(sum(prc_arr)/len(prc_arr))
esp_x = sum(prc_arr)/len(prc_arr)
prc_arr_sq = list(map(lambda x: (x - esp_x)**2, prc_arr))

print(sum(prc_arr_sq)/len(prc_arr_sq))



--- Chunk 1 --- MASK represents 0.4921875
[CLS] educating the world through data [MASK] do [MASK] learn [MASK] [MASK] knowledge [MASK] people [MASK] for? [MASK] [MASK] [MASK] get them that [MASK] and [MASK] [MASK] [MASK]? [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] websites [MASK] over 30, 000 [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK], [MASK] [MASK] [MASK] in [MASK] to generate [MASK] [MASK] answer these questions. [MASK] use [MASK] [MASK] and [MASK] has [MASK] our sites [MASK] has been [MASK] [MASK] our learning as we take more [MASK] in [MASK] our goal [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] globe. gvsu web [MASK] [MASK] " [MASK] [MASK] [MASK] [MASK] [MASK] [MASK] ( 2013 [MASK]. 2013 presentations [MASK] 16 [MASK] this [MASK] is [MASK] not available here.

--- Chunk 2 --- MASK represents 0.53125
moisture [MASK] [MASK] if the top [MASK] [MASK] [MASK] [MASK] [MASK] soil is [MASK], [MASK] [MASK] [MASK] drips [MASK] [MASK] pot into [MASK] saucer [MASK] [MASK] sure to [MASK]